In [1]:
import requests
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class MCPProvider:
    def __init__(self, embedding_model):
        self.embedding_model = embedding_model
        self.sources = []  # Lista de URLs externas confiáveis

    def add_source(self, url):
        """Adiciona uma fonte externa confiável"""
        self.sources.append(url)

    def fetch_pdf_text(self, url):
        """Baixa PDF e extrai o texto"""
        r = requests.get(url)
        with open("temp.pdf", "wb") as f:
            f.write(r.content)
        reader = PdfReader("temp.pdf")
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        return text

    def expand_context(self, milvus_context, top_k=3):
        """Expande o contexto recebido do Milvus com fontes externas"""
        additional_chunks = []
        for url in self.sources:
            try:
                text = self.fetch_pdf_text(url)
                # Divide em trechos de ~300 tokens (aprox.)
                chunks = [text[i:i+1000] for i in range(0, len(text), 1000)]
                
                # Embeddings dos chunks
                chunk_embs = self.embedding_model.encode(chunks)
                milvus_emb = self.embedding_model.encode(milvus_context)
                
                # Calcula similaridade e seleciona os top_k mais relevantes
                sim = cosine_similarity(milvus_emb, chunk_embs)
                top_indices = sim.mean(axis=0).argsort()[-top_k:][::-1]
                
                for idx in top_indices:
                    additional_chunks.append((chunks[idx], url))
            except Exception as e:
                print(f"Erro ao processar {url}: {e}")
        return additional_chunks


In [44]:
# src/mcp_provider/milvus_provider.py
from pymilvus import connections, Collection
from transformers import AutoModel, AutoTokenizer
import torch
import ollama

class MilvusProvider:
    def __init__(self, host="127.0.0.1", port="19530", collection_name="rag_embeddings_milvus"):
        connections.connect("default", host=host, port=port)
        self.collection = Collection(collection_name)

        self.model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModel.from_pretrained(self.model_name)

    def embed_query(self, query: str):
        inputs = self.tokenizer(query, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            outputs = self.model(**inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1)
        return embeddings[0].numpy().tolist()

    def search(self, query: str, top_k: int = 5):
        self.collection.load()
        query_emb = self.embed_query(query)

        results = self.collection.search(
            data=[query_emb],
            anns_field="embedding",
            param={"metric_type": "IP", "params": {"nprobe": 10}},
            limit=top_k,
            output_fields=["source_file", "source_url", "chunk_index", "chunk_text"]
        )

        contexts = []
        refs = []
        for r in results[0]:
            chunk_text = r.entity.get("chunk_text", "")
            source_file = r.entity.get("source_file", "Desconhecido")
            source_url = r.entity.get("source_url", "")
            contexts.append(chunk_text)
            refs.append(f"📄 {source_file} | 🔗 {source_url}")

        return "\n\n".join(contexts), "\n".join(refs)

    def search_images(self, query: str, top_k: int = 5):
        query_emb = self.embed_query(query)
        image_collection = Collection("image_descriptions")
        image_collection.load()

        results = image_collection.search(
            data=[query_emb],
            anns_field="embedding",
            param={"metric_type": "COSINE", "params": {"nprobe": 10}},
            limit=top_k,
            output_fields=["url", "titles", "texts", "category"]
        )

        images = []
        for r in results[0]:
            url = r.entity.get("url")
            titles = r.entity.get("titles")
            texts = r.entity.get("texts")
            category = r.entity.get("category")
            score = r.distance
            images.append(f"🖼️ {category} | {titles} | {texts[:80]}... ({url}) [score={score:.3f}]")

        return "\n".join(images)

    def generate_answer(self, query: str, context: str):
        prompt = f"""
Você é um assistente técnico especializado em licenciamento ambiental (EIA/RIMA).
Responda à pergunta do usuário **usando apenas o contexto fornecido**.

Contexto:
{context}

Pergunta:
{query}

Responda de forma clara, objetiva e técnica.
"""
        response = ollama.chat(
            model="mistral:7b",
            messages=[
                {"role": "system", "content": "Você é um assistente técnico ambiental especializado em EIA/RIMA."},
                {"role": "user", "content": prompt}
            ]
        )
        return response["message"]["content"]

    def ask(self, query: str, top_k: int = 5):
        context, refs = self.search(query, top_k=top_k)
        images = self.search_images(query, top_k=top_k)
        answer = self.generate_answer(query, context)
        return answer, context, refs, images


In [18]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
mcp_provider = MCPProvider(embedding_model)

# Adicione fontes confiáveis (PDFs públicos, sites governamentais, papers)
mcp_provider.add_source("https://cetesb.sp.gov.br/eiarima/eia/EIA-096-24-e-amb-14575-24-Lot-Resid-Jequitiba-Boituva.pdf")
mcp_provider.add_source("https://cetesb.sp.gov.br/eiarima/eia/EIA-058-24-e-amb-4841-23-Ampl-CDR-Pedreira-SP-Capital.pdf")


In [23]:
import ollama

provider = MilvusProvider()

def generate_answer(query: str):
    context, refs = provider.search(query)

    prompt = f"""
Você é um assistente técnico especializado em licenciamento ambiental (EIA/RIMA).
Responda **apenas com base no contexto fornecido**.

Contexto:
{context}

Pergunta:
{query}
"""
    response = ollama.chat(
        model="mistral:7b",
        messages=[
            {"role": "system", "content": "Você é um assistente técnico ambiental especializado em EIA/RIMA."},
            {"role": "user", "content": prompt}
        ]
    )
    print(context)
    return response["message"]["content"], refs


In [46]:
provider = MilvusProvider()
answer, context, refs, images = provider.ask("Como funciona o licenciamento ambiental?")
print(answer)
print(refs)
print(images)



 O Licenciamento Ambiental (LA) é um processo regulamentado pelo governo brasileiro que tem como finalidade avaliar o impacto ambiental de projetos ou atividades antropogênicas, a fim de minimizar danos ao meio ambiente e garantir o desenvolvimento sustentável. Esse processo é previsto nas seguintes normas: Resolução CONAMA 06 (1986), Resolução SMA 09 (2017), Lei n° 4.771/65, Lei n° 6.938/81, Lei n° 10.650/03, entre outras.

No caso específico da ampliação de um empreendimento na cidade de Rio das Pedras (SP), a licença ambiental será solicitada à CETESB e publicada no portal imprensaoficial.com.br. A Companhia de Processamento de Dados do Estado de São Paulo - Prodesp garante a autenticidade deste documento.

Durante o processo de licenciamento ambiental, são analisadas as seguintes áreas: Uso e ocupação do solo, infraestruturas existentes, infraestruturas e serviços públicos, utilização e proteção da vegetação nativa do Bioma Mata Atlântica. A equipe técnica responsável avaliará os i

In [47]:
query = "como uma pedreira afeta o meio ambiente?"
answer, refs = generate_answer(query)

print("🤖", answer)
print("\n--- Fontes ---")
print(refs)


Portanto, trata-se da Ampliação de um Empreendimento em plena atividade, com pequena expansão em área e grande ganho de espaço para a disposição e tratamento finais adequados para aproximadamente 7,4 milhões de toneladas de resíduos sólidos gerados na RMSP e adjacências, onde os principais impactos negativos já ocorreram frente aos usos pretéritos e atuais da área de ocupação Considerando os impactos identificados e avaliados e a adoção das ações de gestão ambiental propostas neste EIA, a equipe técnica responsável pela elaboração deste estudo considera este Empreendimento ambientalmente viável.

• P6 - Projeto de Monitoramento das Áreas Verdes;
A presença de pessoas e o funcionamento de máquinas/equipamentos pode promover perturbação à comunidade de fauna silvestre podendo provocar seu deslocamento e afugentamento, alterando seus hábitos e, desta forma, expô-los a riscos de acidentes e confrontos com funcionários, submetendo-os às condições de estresse.
Como resultado do afugentamento

In [9]:
import requests
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm  # Para feedback visual do processamento

class MCPProvider:
    def __init__(self, embedding_model):
        self.embedding_model = embedding_model
        self.sources = []  # Lista de URLs externas confiáveis

    def add_source(self, url):
        """Adiciona uma fonte externa confiável"""
        self.sources.append(url)

    def fetch_pdf_text(self, url):
        """Baixa PDF e extrai o texto"""
        print(f"📥 Baixando PDF: {url}")
        r = requests.get(url)
        with open("temp.pdf", "wb") as f:
            f.write(r.content)
        reader = PdfReader("temp.pdf")
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        print(f"✅ PDF processado: {url}")
        return text

    def expand_context(self, milvus_context, top_k=3):
        """Expande o contexto recebido do Milvus com fontes externas"""
        print("🔍 Iniciando expansão de contexto com MCP...")
        additional_chunks = []

        # Cria embeddings do contexto do Milvus uma vez
        print("🧠 Gerando embeddings do contexto do Milvus...")
        milvus_emb = self.embedding_model.encode(milvus_context)

        for url in tqdm(self.sources, desc="🌐 Processando fontes externas"):
            try:
                text = self.fetch_pdf_text(url)
                # Divide em trechos de ~1000 caracteres (aprox. 300 tokens)
                chunks = [text[i:i+1000] for i in range(0, len(text), 1000)]

                # Embeddings dos chunks
                print(f"🧮 Gerando embeddings para os chunks de {url}...")
                chunk_embs = self.embedding_model.encode(chunks)

                # Calcula similaridade e seleciona os top_k mais relevantes
                print(f"📊 Calculando similaridade para {url}...")
                sim = cosine_similarity(milvus_emb, chunk_embs)
                top_indices = sim.mean(axis=0).argsort()[-top_k:][::-1]

                for idx in top_indices:
                    additional_chunks.append((chunks[idx], url))
                
                print(f"✅ Top {top_k} chunks adicionados de {url}\n")
            except Exception as e:
                print(f"⚠️ Erro ao processar {url}: {e}")

        print("🎯 Expansão de contexto concluída!")
        return additional_chunks


# ------------------------------
# Inicialização
# ------------------------------
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
mcp_provider = MCPProvider(embedding_model)

# Adicione fontes confiáveis (PDFs públicos, sites governamentais, papers)
mcp_provider.add_source("https://philip.inpa.gov.br/publ_livres/Dossie/S_Manoel/Docs_of/EIA/EIA%20UHE%20Sao%20Manoel%20-%20Volume_1.pdf")
mcp_provider.add_source("http://www.uhecastanheira.com.br/wp-content/uploads/bigfiles/02_EIA_Capitulo_08_ao_Capitulo_13_BR.pdff")


# ------------------------------
# Fluxo da pergunta
# ------------------------------
query = "Quais são os impactos de barragens hidrelétricas?"

# 1️⃣ Milvus
milvus_context, milvus_refs = provider.search(query)

# 2️⃣ MCP expande contexto
mcp_results = mcp_provider.expand_context(milvus_context)

# 3️⃣ Junta contextos
full_context = milvus_context + [chunk for chunk, url in mcp_results]
all_refs = milvus_refs + [url for chunk, url in mcp_results]

# 4️⃣ Gera resposta com LLM
prompt = f"""
Você é um assistente técnico especializado em licenciamento ambiental (EIA/RIMA).
Use apenas o contexto fornecido para responder a pergunta.

Contexto:
{full_context}

Pergunta:
{query}
"""

response = ollama.chat(
    model="mistral:7b",
    messages=[{"role": "user", "content": prompt}]
)


answer = response["message"]["content"]


print("\n🤖 Resposta:")
print(answer)
print("\n--- Fontes ---")
print(all_refs)

print(full_context)


🔍 Iniciando expansão de contexto com MCP...
🧠 Gerando embeddings do contexto do Milvus...


🌐 Processando fontes externas:   0%|                                                            | 0/2 [00:00<?, ?it/s]

📥 Baixando PDF: https://philip.inpa.gov.br/publ_livres/Dossie/S_Manoel/Docs_of/EIA/EIA%20UHE%20Sao%20Manoel%20-%20Volume_1.pdf
✅ PDF processado: https://philip.inpa.gov.br/publ_livres/Dossie/S_Manoel/Docs_of/EIA/EIA%20UHE%20Sao%20Manoel%20-%20Volume_1.pdf
🧮 Gerando embeddings para os chunks de https://philip.inpa.gov.br/publ_livres/Dossie/S_Manoel/Docs_of/EIA/EIA%20UHE%20Sao%20Manoel%20-%20Volume_1.pdf...


🌐 Processando fontes externas:  50%|██████████████████████████                          | 1/2 [01:08<01:08, 68.40s/it]

📊 Calculando similaridade para https://philip.inpa.gov.br/publ_livres/Dossie/S_Manoel/Docs_of/EIA/EIA%20UHE%20Sao%20Manoel%20-%20Volume_1.pdf...
✅ Top 3 chunks adicionados de https://philip.inpa.gov.br/publ_livres/Dossie/S_Manoel/Docs_of/EIA/EIA%20UHE%20Sao%20Manoel%20-%20Volume_1.pdf

📥 Baixando PDF: http://www.uhecastanheira.com.br/wp-content/uploads/bigfiles/02_EIA_Capitulo_08_ao_Capitulo_13_BR.pdff


🌐 Processando fontes externas: 100%|████████████████████████████████████████████████████| 2/2 [01:15<00:00, 37.62s/it]

⚠️ Erro ao processar http://www.uhecastanheira.com.br/wp-content/uploads/bigfiles/02_EIA_Capitulo_08_ao_Capitulo_13_BR.pdff: EOF marker not found
🎯 Expansão de contexto concluída!



🤖 Resposta:
 Os impactos de barragens hidrelétricas podem incluir:

1. Alteração do ambiente natural, como modificação do curso de rios e bacias hidrográficas, alterações nas áreas de inundação temporária e permanente, e alterações na vegetação e fauna da região.
2. Efeitos sobre a qualidade da água, como aumento da sedimentação, mudanças na concentração de nutrientes e contaminantes, e alterações na temperatura e oxigenação do água.
3. Impactos sociais e culturais, como deslocamento de populações locais, alteração dos costumes tradicionais, e impacto sobre o patrimônio cultural local.
4. Alteração do equilíbrio geológico, como aumento da pressão hidráulica na rocha subjacente e possibilidade de deslizamentos de terra ou rochas.
5. Impactos ambientais associados à construção da barragem, como poluição do ar devido a geração de poeira durante o processo de engenharia civil, e impactos sobre os recursos minerais locais.
6. Impactos energéticos, como mudanças na distribuição e disponibil